# 03_features

**Input:** `rides_clean.parquet` (built by notebook 02)

**Output:** `rides_features.parquet`, the cleaned rides plus 6 new columns for analysis

**Why this exists:** the raw data has timestamps, but the analysis needs rides grouped by day, hour and season, so this notebook creates: ride_min, day_of_week, day_type, hour, month, season

In [2]:
import pandas as pd
import numpy as np

rides = pd.read_parquet("../data/clean/rides_clean.parquet")

days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
season_map = {12: "Winter", 1: "Winter", 2: "Winter",
              3: "Spring", 4: "Spring", 5: "Spring",
              6: "Summer", 7: "Summer", 8: "Summer",
              9: "Fall", 10: "Fall", 11: "Fall"}

rides["ride_min"] = rides["dur_sec"] / 60
rides["day_of_week"] = pd.Categorical(rides["started_at"].dt.day_name(), categories=days, ordered=True)
rides["day_type"] = np.where(rides["started_at"].dt.day_of_week >= 5, "Weekend", "Weekday")
rides["hour"] = rides["started_at"].dt.hour.astype("int8")
rides["month"] = rides["started_at"].dt.to_period("M").astype(str)
rides["season"] = rides["started_at"].dt.month.map(season_map)

rides.to_parquet("../data/clean/rides_features.parquet")

print(rides[["started_at", "ride_min", "day_of_week", "day_type", "hour", "month", "season"]].head())
print(rides["season"].value_counts())
print(rides["day_type"].value_counts())

               started_at   ride_min day_of_week day_type  hour    month  \
0 2025-10-27 08:19:53.208   6.004550      Monday  Weekday     8  2025-10   
1 2025-10-14 07:42:55.465  21.219967     Tuesday  Weekday     7  2025-10   
2 2025-10-12 08:18:23.613  10.256167      Sunday  Weekend     8  2025-10   
3 2025-10-14 07:45:56.471  21.466267     Tuesday  Weekday     7  2025-10   
4 2025-10-19 10:08:45.138   3.827217      Sunday  Weekend    10  2025-10   

  season  
0   Fall  
1   Fall  
2   Fall  
3   Fall  
4   Fall  
season
Summer    2433903
Fall      1726393
Spring    1379339
Winter     467519
Name: count, dtype: int64
day_type
Weekday    4315991
Weekend    1691163
Name: count, dtype: int64
